In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json
from glob import glob
from scipy.stats import pearsonr
import seaborn as sns

In [ ]:
for file in glob("/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn*/session00_7.npy"):
    print(file, np.load(file).shape)

In [ ]:
with open("/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn_eso245_aal_strin_90_noRank_7nn/cmiknn_eso245_aal_strin_90_noRank_7nn_globalStats.json")    as fp:
    knnStats_noRank = {k: np.array(v) for k, v in json.load(fp).items()}
with open("/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn_eso245_aal_strin_90_noRank_EC_7nn/cmiknn_eso245_aal_strin_90_noRank_EC_7nn_globalStats.json")    as fp:
    knnEC_noRank = {k: np.array(v) for k, v in json.load(fp).items()}
with open("/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn_eso245_aal_strin_90_7nn/cmiknn_eso245_aal_strin_90_7nn_globalStats.json")    as fp:
    knnStats = {k: np.array(v) for k, v in json.load(fp).items()}
with open("/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn_eso245_aal_strin_90_EC_7nn/cmiknn_eso245_aal_strin_90_EC_7nn_globalStats.json")    as fp:
    knnEC = {k: np.array(v) for k, v in json.load(fp).items()}
with open("/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/eso245_aal_strin_90_bin7/eso245_aal_strin_90_bin7_globalStats.json")    as fp:
    binStats = {k: np.array(v) for k, v in json.load(fp).items()}

In [ ]:
CMI = knnEC["mean"]
CMI_shadow = knnEC["meanshadow"]
CMI_noRank = knnEC_noRank["mean"]
CMI_noRank_shadow = knnEC_noRank["meanshadow"]
binRNL = (binStats['totalMI']-binStats['gaussMI'])#/binStats['totalMI']
binRNL_shadow = (binStats['totalMIshadow']-binStats['gaussMIshadow'])#/binStats['totalMIshadow']
knnRNL = (knnStats['totalMI']-knnStats['gaussMI'])#/knnStats['totalMI']
knnRNL_shadow = (knnStats['totalMIshadow']-knnStats['gaussMIshadow'])#/knnStats['totalMIshadow']
knnRNL_noRank = (knnStats_noRank['totalMI']-knnStats_noRank['gaussMI'])#/knnStats_noRank['totalMI']
knnRNL_noRank_shadow = (knnStats_noRank['totalMIshadow']-knnStats_noRank['gaussMIshadow'])#/knnStats_noRank['totalMIshadow']

In [ ]:
y_list = ["CMI", "CMI_noRank", "binRNL"]
x_list = ["knnRNL", "knnRNL_noRank", "binRNL"]
for y_name in y_list:
    for x_name in x_list:
        if x_name == y_name:
            continue
        x = eval(x_name)
        y = eval(y_name)
        print(y_name, x_name, pearsonr(x, y))
        x_shadow = eval(x_name + "_shadow")
        y_shadow = eval(y_name + "_shadow")
        print("shadow", pearsonr(x_shadow, y_shadow))

        plt.scatter(x, y, label="Empirical")
        plt.scatter(x_shadow, y_shadow, marker="^", label="Shadow")
        cg=0
        rg=0
        for c, cs, r, rs in zip(x, x_shadow, y, y_shadow):
            lw=1
            o="g"
            if cs > c:
                o="b"
                cg+=1
                lw+=1
            if rs > r:
                o = "r" if o == "g" else "k"
                rg+=1
                lw+=1
            plt.plot([c, cs], [r, rs], o+":", lw=lw, zorder=-1)
        plt.xlabel(x_name)
        plt.ylabel(y_name)
        plt.legend()
        plt.title(f"#{x_name}<{x_name}_shadow = {cg}, #{y_name}<{y_name}_shadow = {rg}")
        plt.show()
        thresholdsMI = sorted(np.concatenate([x, x_shadow]), reverse=True)
        thresholdsEC = sorted(np.concatenate([y, y_shadow]), reverse=True)
        TP_MI = []
        FP_MI = []
        TP_EC = []
        FP_EC = []
        for t in thresholdsMI:
            TP_MI.append(np.sum(x > t)/245)
            FP_MI.append(np.sum(x_shadow > t)/245)
        for t in thresholdsEC:
            TP_EC.append(np.sum(y > t)/245)
            FP_EC.append(np.sum(y_shadow > t)/245)
        plt.plot(FP_MI, TP_MI, label=f"{x_name}, auc:{(np.diff(FP_MI)*TP_MI[1:]).sum():.2}")
        plt.plot(FP_EC, TP_EC, label=f"{y_name}, auc:{(np.diff(FP_EC)*TP_EC[1:]).sum():.2}")
        plt.plot([0,1],[0,1], "k--")
        plt.legend()
        plt.xlabel("False Positives")
        plt.ylabel("True Positives")
        plt.title(f"{x_name} vs {y_name}")
        plt.xlim(0,1)
        plt.ylim(0,1)
        plt.show()

In [ ]:
def coloured_box_kwargs(colour):
    return {
        "boxprops": {"color": colour},
        # "widths": 0.6,
        "whiskerprops": {"color": colour},
        "flierprops": {"markeredgecolor": colour, "marker": "+"},
        "medianprops": {"lw": 1.6, "color": colour},
        "capprops": {"color": colour},
    }

In [ ]:
empirical = []
shadow = []
for rnl in ["knnRNL", "knnRNL_noRank", "binRNL"]:
    empirical.append(eval(rnl))
    shadow.append(eval(rnl + "_shadow"))
plt.boxplot(empirical, positions=[1, 2, 3], widths=0.4, **coloured_box_kwargs(sns.color_palette("colorblind")[1]))
plt.boxplot(shadow, positions=[1.5, 2.5, 3.5], widths=0.4, **coloured_box_kwargs(sns.color_palette("colorblind")[0]))
plt.xticks([1.25, 2.25, 3.25], ["knnRNL", "knnRNL_noRank", "binRNL"])
plt.xlim(plt.xlim())
plt.hlines([0,0.05], *plt.xlim(), "r", linestyles=["solid", "dashed"])
plt.show()

In [ ]:
x = binRNL
y = knnRNL
print(pearsonr(x, y))
x = binRNL_shadow
y = knnRNL_shadow
print(pearsonr(x, y))
x = CMI
y = knnRNL[:len(CMI)]
print(pearsonr(x, y))
x = CMI_shadow
y = knnRNL_shadow
print(pearsonr(x, y))

In [ ]:
x = knnStats["totalMI"]
y = binStats["totalMI"]
print(pearsonr(x, y))
x = knnStats["gaussMI"]
y = binStats["gaussMI"]
print(pearsonr(x, y))
x = knnStats["gaussMI"]/knnStats["totalMI"]
y = binStats["gaussMI"]/binStats["totalMI"]
print(pearsonr(x, y))

x = knnStats_noRank["totalMI"]
y = binStats["totalMI"]
print(pearsonr(x, y))
x = knnStats_noRank["gaussMI"]
y = binStats["gaussMI"]
print(pearsonr(x, y))

# Norm distance for all

In [ ]:
import sys
sys.path.append("../")
from mienc.corrector import Corrector
from mienc.support import get_pool, statistics
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

In [ ]:
corrector = Corrector(
            7,
            duration=400,
            folder_name="/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn_eso245_aal_strin_90_noRank_7nn",
            cache_dir="/mnt/DATA/Giulio_NonLinearityResults/ReRun/cache",
            workers=24,
            display=False,
            retrieve=True,
            config="/mnt/DATA/Giulio_NonLinearityResults/ReRun/config.ini",
            verbose=True,
            no_correction=True
        )
corrector.compute_correction()
polla = get_pool(24)

In [ ]:
corrector = Corrector(
            7,
            duration=400,
            folder_name="/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/eso245_aal_strin_90_bin7",
            cache_dir="/mnt/DATA/Giulio_NonLinearityResults/ReRun/cache",
            workers=24,
            display=False,
            retrieve=True,
            config="/mnt/DATA/Giulio_NonLinearityResults/ReRun/config.ini",
            verbose=True,
            no_correction=False
        )
corrector.compute_correction()
for subject in tqdm(range(245), desc="precorr"):
    data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/eso245_aal_strin_90_bin7/subject{subject:02}_7.npy")
    np.save(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/eso245_aal_strin_90_bin7/subject{subject:02}_7_precorr.npy", corrector.correct(data), )

In [ ]:
distances = {}
distances_shadow = {}
with get_pool(24) as polla:
    for k1, v1 in {"EC":"_EC", "MI":""}.items():
        for k2, v2 in {"noRank": "_noRank", "Rank": ""}.items():
            distances[k1+k2] = []
            distances_shadow[k1+k2] = []
            for subject in tqdm(range(245), desc=k1+k2):
                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn_eso245_aal_strin_90{v2}{v1}_7nn/session{subject:02}_7.npy")
                if k1 == "MI":
                    tmp = np.zeros([90,90, 100])
                    tmp[*np.triu_indices(90,1),:]=data
                    tmp = tmp + tmp.swapaxes(0,1)
                    data = tmp
                distances[k1+k2].append(statistics(data, False, True, polla, corrector)["mean"])

                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn_eso245_aal_strin_90{v2}{v1}_7nn/session{subject:02}_7_sha.npy")
                if k1 == "MI":
                    tmp = np.zeros([90,90, 100])
                    tmp[*np.triu_indices(90,1),:]=data
                    tmp = tmp + tmp.swapaxes(0,1)
                    data = tmp
                distances_shadow[k1+k2].append(statistics(data, False, True, polla, corrector)["mean"])

        if k1 == "MI":
            k2 = "bin"
            distances[k1+k2] = []
            distances_shadow[k1+k2] = []
            corrector = Corrector(
                        7,
                        duration=400,
                        folder_name="/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn_eso245_aal_strin_90_noRank_bin7",
                        cache_dir="/mnt/DATA/Giulio_NonLinearityResults/ReRun/cache",
                        workers=24,
                        display=False,
                        retrieve=True,
                        config="/mnt/DATA/Giulio_NonLinearityResults/ReRun/config.ini",
                        verbose=True,
                        no_correction=False
                    )
            corrector.compute_correction()
            for subject in tqdm(range(245), desc=k1+k2):
                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/eso245_aal_strin_90_bin7/subject{subject:02}_7.npy")
                tmp = np.zeros([90,90, 100])
                tmp[*np.triu_indices(90,1),:]=data
                tmp = tmp + tmp.swapaxes(0,1)
                data = tmp
                distances[k1+k2].append(statistics(data, False, True, polla, corrector)["mean"])

                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/eso245_aal_strin_90_bin7/subject{subject:02}_7_sha.npy")
                tmp = np.zeros([90,90, 100])
                tmp[*np.triu_indices(90,1),:]=data
                tmp = tmp + tmp.swapaxes(0,1)
                data = tmp
                distances_shadow[k1+k2].append(statistics(data, False, True, polla, corrector)["mean"])

In [ ]:
for which in ["Rank", "noRank"]:
    x = distances["MIbin"]
    y = distances["MI"+which]
    x_name = "MI binning"
    y_name = "MI"
    print(y_name, x_name, pearsonr(x, y))
    x_shadow = distances_shadow["EC"+which]
    y_shadow = distances_shadow["MI"+which]
    print("shadow", pearsonr(x_shadow, y_shadow))

    plt.scatter(x, y, label="Empirical")
    plt.scatter(x_shadow, y_shadow, marker="^", label="Shadow")
    cg=0
    rg=0
    for c, cs, r, rs in zip(x, x_shadow, y, y_shadow):
        lw=1
        o="g"
        if cs > c:
            o="b"
            cg+=1
            lw+=1
        if rs > r:
            o = "r" if o == "g" else "k"
            rg+=1
            lw+=1
        plt.plot([c, cs], [r, rs], o+":", lw=lw, zorder=-1)
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    plt.legend()
    plt.title(f"{which} - #{x_name}<{x_name}_shadow = {cg}, #{y_name}<{y_name}_shadow = {rg}")
    plt.show()
    thresholdsMI = sorted(np.concatenate([distances["MI"+which], distances_shadow["MI"+which]]), reverse=True)
    thresholdsEC = sorted(np.concatenate([distances["EC"+which], distances_shadow["EC"+which]]), reverse=True)
    TP_MI = []
    FP_MI = []
    TP_EC = []
    FP_EC = []
    for t in thresholdsMI:
        TP_MI.append(np.sum(distances["MI"+which] > t)/245)
        FP_MI.append(np.sum(distances_shadow["MI"+which] > t)/245)
    for t in thresholdsEC:
        TP_EC.append(np.sum(distances["EC"+which] > t)/245)
        FP_EC.append(np.sum(distances_shadow["EC"+which] > t)/245)
    plt.plot(FP_MI, TP_MI, label=f"MI, auc:{(np.diff(FP_MI)*TP_MI[1:]).sum():.2}")
    plt.plot(FP_EC, TP_EC, label=f"EC, auc:{(np.diff(FP_EC)*TP_EC[1:]).sum():.2}")
    plt.plot([0,1],[0,1], "k--")
    plt.legend()
    plt.xlabel("False Positives")
    plt.ylabel("True Positives")
    plt.title(f"{which}")
    plt.xlim(0,1)
    plt.ylim(0,1)
    plt.show()

In [ ]:
for which in ["Rank", "noRank"]:
    x = distances["EC"+which]
    y = distances["MI"+which]
    x_name = "EC"
    y_name = "MI"
    print(y_name, x_name, pearsonr(x, y))
    x_shadow = distances_shadow["EC"+which]
    y_shadow = distances_shadow["MI"+which]
    print("shadow", pearsonr(x_shadow, y_shadow))

    plt.scatter(x, y, label="Empirical")
    plt.scatter(x_shadow, y_shadow, marker="^", label="Shadow")
    cg=0
    rg=0
    for c, cs, r, rs in zip(x, x_shadow, y, y_shadow):
        lw=1
        o="g"
        if cs > c:
            o="b"
            cg+=1
            lw+=1
        if rs > r:
            o = "r" if o == "g" else "k"
            rg+=1
            lw+=1
        plt.plot([c, cs], [r, rs], o+":", lw=lw, zorder=-1)
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    plt.legend()
    plt.title(f"{which} - #{x_name}<{x_name}_shadow = {cg}, #{y_name}<{y_name}_shadow = {rg}")
    plt.show()
    thresholdsMI = sorted(np.concatenate([distances["MI"+which], distances_shadow["MI"+which]]), reverse=True)
    thresholdsEC = sorted(np.concatenate([distances["EC"+which], distances_shadow["EC"+which]]), reverse=True)
    TP_MI = []
    FP_MI = []
    TP_EC = []
    FP_EC = []
    for t in thresholdsMI:
        TP_MI.append(np.sum(distances["MI"+which] > t)/245)
        FP_MI.append(np.sum(distances_shadow["MI"+which] > t)/245)
    for t in thresholdsEC:
        TP_EC.append(np.sum(distances["EC"+which] > t)/245)
        FP_EC.append(np.sum(distances_shadow["EC"+which] > t)/245)
    plt.plot(FP_MI, TP_MI, label=f"MI, auc:{(np.diff(FP_MI)*TP_MI[1:]).sum():.2}")
    plt.plot(FP_EC, TP_EC, label=f"EC, auc:{(np.diff(FP_EC)*TP_EC[1:]).sum():.2}")
    plt.plot([0,1],[0,1], "k--")
    plt.legend()
    plt.xlabel("False Positives")
    plt.ylabel("True Positives")
    plt.title(f"{which}")
    plt.xlim(0,1)
    plt.ylim(0,1)
    plt.show()

In [ ]:
distances = {}
distances_shadow = {}
with get_pool(24) as polla:
    for k1 in ["raw", "mod", "strin"]:
        if k1 == "strin":
            for k2, v2 in {"noRank": "_noRank", "Rank": ""}.items():
                distances[k1+k2] = []
                distances_shadow[k1+k2] = []
                for subject in tqdm(range(245), desc=k1+k2):
                    data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn_eso245_aal_{k1}_90{v2}_7nn/session{subject:02}_7.npy")
                    tmp = np.zeros([90,90, 100])
                    tmp[*np.triu_indices(90,1),:]=data
                    tmp = tmp + tmp.swapaxes(0,1)
                    data = tmp
                    distances[k1+k2].append(statistics(data, False, True, polla, corrector)["mean"])

                    data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/cmiknn_eso245_aal_{k1}_90{v2}_7nn/session{subject:02}_7_sha.npy")
                    tmp = np.zeros([90,90, 100])
                    tmp[*np.triu_indices(90,1),:]=data
                    tmp = tmp + tmp.swapaxes(0,1)
                    data = tmp
                    distances_shadow[k1+k2].append(statistics(data, False, True, polla, corrector)["mean"])
        for k2 in {"dChat"}:
            distances[k1+k2] = []
            distances_shadow[k1+k2] = []
            for subject in tqdm(range(245), desc=k1+k2):
                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/eso245_aal_{k1}_90_EC_t0/session{subject:02}_0.npy")
                distances[k1+k2].append(statistics(data, False, True, polla, None)["mean"])

                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/eso245_aal_{k1}_90_EC_t0/session{subject:02}_0_sha.npy")
                distances_shadow[k1+k2].append(statistics(data, False, True, polla, None)["mean"])

In [ ]:
for which in ["Rank", "noRank"]:
    for k2 in ["raw", "mod", "strin"]:
        x = distances[k2+"dChat"]
        y = distances["strin"+which]
        x_name = "dChatterjee"
        y_name = "MI"
        print(y_name, x_name, pearsonr(x, y))
        x_shadow = distances_shadow[k2+"dChat"]
        y_shadow = distances_shadow["strin"+which]
        print("shadow", pearsonr(x_shadow, y_shadow))

        plt.scatter(x, y, label="Empirical")
        plt.scatter(x_shadow, y_shadow, marker="^", label="Shadow")
        cg=0
        rg=0
        for c, cs, r, rs in zip(x, x_shadow, y, y_shadow):
            lw=1
            o="g"
            if cs > c:
                o="b"
                cg+=1
                lw+=1
            if rs > r:
                o = "r" if o == "g" else "k"
                rg+=1
                lw+=1
            plt.plot([c, cs], [r, rs], o+":", lw=lw, zorder=-1)
        plt.xlabel(x_name)
        plt.ylabel(y_name)
        plt.legend()
        plt.title(f"{which} - {k2} - #{x_name}<{x_name}_shadow = {cg}, #{y_name}<{y_name}_shadow = {rg}")
        plt.show()
        thresholdsMI = sorted(np.concatenate([distances["strin"+which], distances_shadow["strin"+which]]), reverse=True)
        thresholdsEC = sorted(np.concatenate([distances[k2+"dChat"], distances_shadow[k2+"dChat"]]), reverse=True)
        TP_MI = []
        FP_MI = []
        TP_EC = []
        FP_EC = []
        for t in thresholdsMI:
            TP_MI.append(np.sum(distances["strin"+which] > t)/245)
            FP_MI.append(np.sum(distances_shadow["strin"+which] > t)/245)
        for t in thresholdsEC:
            TP_EC.append(np.sum(distances[k2+"dChat"] > t)/245)
            FP_EC.append(np.sum(distances_shadow[k2+"dChat"] > t)/245)
        plt.plot(FP_MI, TP_MI, label=f"MI, auc:{(np.diff(FP_MI)*TP_MI[1:]).sum():.2}")
        plt.plot(FP_EC, TP_EC, label=f"dC, auc:{(np.diff(FP_EC)*TP_EC[1:]).sum():.2}")
        plt.plot([0,1],[0,1], "k--")
        plt.legend()
        plt.xlabel("False Positives")
        plt.ylabel("True Positives")
        plt.title(f"{which} - {k2}")
        plt.xlim(0,1)
        plt.ylim(0,1)
        plt.show()

In [ ]:
distances = {}
distances_shadow = {}
with get_pool(24) as polla:
    for band, bin in zip(["delta", "theta", "alpha", "beta", "gamma"], [10, 13, 14, 20, 23]):
        for k2 in {"MI"}:
            distances[band+k2] = []
            distances_shadow[band+k2] = []
            for subject in tqdm(range(150), desc=band+k2):
                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/EEG_124s_band_{band}_bin{bin}/subject{subject:02}_{bin}.npy")
                tmp = np.zeros([54,54, 100])
                tmp[*np.triu_indices(54,1),:]=data
                tmp = tmp + tmp.swapaxes(0,1)
                data = tmp
                distances[band+k2].append(statistics(data, False, True, polla, corrector)["mean"])

                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/EEG_124s_band_{band}_bin{bin}/subject{subject:02}_{bin}_sha.npy")
                tmp = np.zeros([54,54, 100])
                tmp[*np.triu_indices(54,1),:]=data
                tmp = tmp + tmp.swapaxes(0,1)
                data = tmp
                distances_shadow[band+k2].append(statistics(data, False, True, polla, corrector)["mean"])
        for k2 in {"dChat"}:
            distances[band+k2] = []
            distances_shadow[band+k2] = []
            for subject in tqdm(range(150), desc=band+k2):
                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/EEG_124s_band_{band}_EC_t0/session{subject:02}_0.npy")
                distances[band+k2].append(statistics(data, False, True, polla, None)["mean"])

                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/EEG_124s_band_{band}_EC_t0/session{subject:02}_0_sha.npy")
                distances_shadow[band+k2].append(statistics(data, False, True, polla, None)["mean"])

In [ ]:
for which in ["delta", "theta", "alpha", "beta", "gamma"]:
    x = distances[which+"dChat"]
    y = distances[which+"MI"]
    x_name = "dChatterjee"
    y_name = "MI"
    print(y_name, x_name, pearsonr(x, y))
    x_shadow = distances_shadow[which+"dChat"]
    y_shadow = distances_shadow[which+"MI"]
    print("shadow", pearsonr(x_shadow, y_shadow))

    plt.scatter(x, y, label="Empirical")
    plt.scatter(x_shadow, y_shadow, marker="^", label="Shadow")
    cg=0
    rg=0
    for c, cs, r, rs in zip(x, x_shadow, y, y_shadow):
        lw=1
        o="g"
        if cs > c:
            o="b"
            cg+=1
            lw+=1
        if rs > r:
            o = "r" if o == "g" else "k"
            rg+=1
            lw+=1
        plt.plot([c, cs], [r, rs], o+":", lw=lw, zorder=-1)
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    plt.legend()
    plt.title(f"{which} - {k2} - #{x_name}<{x_name}_shadow = {cg}, #{y_name}<{y_name}_shadow = {rg}")
    plt.show()
    thresholdsMI = sorted(np.concatenate([distances[which+"MI"], distances_shadow[which+"MI"]]), reverse=True)
    thresholdsEC = sorted(np.concatenate([distances[which+"dChat"], distances_shadow[which+"dChat"]]), reverse=True)
    TP_MI = []
    FP_MI = []
    TP_EC = []
    FP_EC = []
    for t in thresholdsMI:
        TP_MI.append(np.sum(distances[which+"MI"] > t)/150)
        FP_MI.append(np.sum(distances_shadow[which+"MI"] > t)/150)
    for t in thresholdsEC:
        TP_EC.append(np.sum(distances[which+"dChat"] > t)/150)
        FP_EC.append(np.sum(distances_shadow[which+"dChat"] > t)/150)
    plt.plot(FP_MI, TP_MI, label=f"MI, auc:{(np.diff(FP_MI)*TP_MI[1:]).sum():.2}")
    plt.plot(FP_EC, TP_EC, label=f"dC, auc:{(np.diff(FP_EC)*TP_EC[1:]).sum():.2}")
    plt.plot([0,1],[0,1], "k--")
    plt.legend()
    plt.xlabel("False Positives")
    plt.ylabel("True Positives")
    plt.title(f"{which} - {k2}")
    plt.xlim(0,1)
    plt.ylim(0,1)
    plt.show()

In [ ]:
distances = {}
distances_shadow = {}
with get_pool(24) as polla:
    for band, bin in zip(["delta", "theta", "alpha", "beta", "gamma"], [10, 13, 14, 20, 23]):
        for k2 in {"MI"}:
            distances[band+k2] = []
            distances_shadow[band+k2] = []
            for subject in tqdm(range(18), desc=band+k2):
                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/iEEG_ses01_band_{band}_bin{bin}/subject{subject:02}_{bin}.npy")
                tmp = np.zeros([24,24, 100])
                tmp[*np.triu_indices(24,1),:]=data
                tmp = tmp + tmp.swapaxes(0,1)
                data = tmp
                distances[band+k2].append(statistics(data, False, True, polla, corrector)["mean"])

                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/iEEG_ses01_band_{band}_bin{bin}/subject{subject:02}_{bin}_sha.npy")
                tmp = np.zeros([24,24, 100])
                tmp[*np.triu_indices(24,1),:]=data
                tmp = tmp + tmp.swapaxes(0,1)
                data = tmp
                distances_shadow[band+k2].append(statistics(data, False, True, polla, corrector)["mean"])
        for k2 in {"dChat"}:
            distances[band+k2] = []
            distances_shadow[band+k2] = []
            for subject in tqdm(range(18), desc=band+k2):
                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/iEEG_ses01_band_{band}_EC_t0/session{subject:02}_0.npy")
                distances[band+k2].append(statistics(data, False, True, polla, None)["mean"])

                data = np.load(f"/mnt/DATA/Giulio_NonLinearityResults/ReRun/Results/iEEG_ses01_band_{band}_EC_t0/session{subject:02}_0_sha.npy")
                distances_shadow[band+k2].append(statistics(data, False, True, polla, None)["mean"])

In [ ]:
for which in ["delta", "theta", "alpha", "beta", "gamma"]:
    x = distances[which+"dChat"]
    y = distances[which+"MI"]
    x_name = "dChatterjee"
    y_name = "MI"
    print(y_name, x_name, pearsonr(x, y))
    x_shadow = distances_shadow[which+"dChat"]
    y_shadow = distances_shadow[which+"MI"]
    print("shadow", pearsonr(x_shadow, y_shadow))

    plt.scatter(x, y, label="Empirical")
    plt.scatter(x_shadow, y_shadow, marker="^", label="Shadow")
    cg=0
    rg=0
    for c, cs, r, rs in zip(x, x_shadow, y, y_shadow):
        lw=1
        o="g"
        if cs > c:
            o="b"
            cg+=1
            lw+=1
        if rs > r:
            o = "r" if o == "g" else "k"
            rg+=1
            lw+=1
        plt.plot([c, cs], [r, rs], o+":", lw=lw, zorder=-1)
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    plt.legend()
    plt.title(f"iEEG - {which} - {k2} - #{x_name}<{x_name}_shadow = {cg}, #{y_name}<{y_name}_shadow = {rg}")
    plt.show()
    thresholdsMI = sorted(np.concatenate([distances[which+"MI"], distances_shadow[which+"MI"]]), reverse=True)
    thresholdsEC = sorted(np.concatenate([distances[which+"dChat"], distances_shadow[which+"dChat"]]), reverse=True)
    TP_MI = []
    FP_MI = []
    TP_EC = []
    FP_EC = []
    for t in thresholdsMI:
        TP_MI.append(np.sum(distances[which+"MI"] > t)/18)
        FP_MI.append(np.sum(distances_shadow[which+"MI"] > t)/18)
    for t in thresholdsEC:
        TP_EC.append(np.sum(distances[which+"dChat"] > t)/18)
        FP_EC.append(np.sum(distances_shadow[which+"dChat"] > t)/18)
    plt.plot(FP_MI, TP_MI, label=f"MI, auc:{(np.diff(FP_MI)*TP_MI[1:]).sum():.2}")
    plt.plot(FP_EC, TP_EC, label=f"dC, auc:{(np.diff(FP_EC)*TP_EC[1:]).sum():.2}")
    plt.plot([0,1],[0,1], "k--")
    plt.legend()
    plt.xlabel("False Positives")
    plt.ylabel("True Positives")
    plt.title(f"iEEG - {which} - {k2}")
    plt.xlim(0,1)
    plt.ylim(0,1)
    plt.show()